## Estimadores Puntuales

### Definición

Un **estimador** es una función de la muestra que se usa para aproximar un parámetro poblacional desconocido.

$$\hat{\theta} = h(X_1, X_2, \dots, X_n)$$

Donde $\theta$ es el parámetro poblacional y $\hat{\theta}$ es su estimador.

###  Analogía

> "El estimador es como **adivinar el peso de un pez** basándote en una muestra de peces. No sabes el peso promedio real (parámetro), pero usas el promedio de la muestra (estimador) para aproximarlo."

###  Tipos de Estimadores

| Tipo | Descripción | Ejemplo |
|------|-------------|---------|
| **Estimador puntual** | Un solo valor | $\bar{x} = 5.2$ |
| **Estimador por intervalo** | Un rango de valores | $[4.8, 5.6]$ |

###  Propiedades deseables de un estimador

**1. Insesgadez (Sesgo = 0)**

$$Bias(\hat{\theta}) = \mathbb{E}[\hat{\theta}] - \theta = 0$$

- El estimador **acierta en promedio**
- Ejemplo: $\bar{X}$ es insesgado para $\mu$

**2. Eficiencia (Mínima Varianza)**

$$Var(\hat{\theta}_1) < Var(\hat{\theta}_2)$$

- Entre dos estimadores insesgados, es mejor el de menor varianza
- El **MEI** (Mejor Estimador Insesgado) tiene varianza mínima

**3. Consistencia**

$$\hat{\theta}_n \xrightarrow{p} \theta \quad \text{cuando } n \to \infty$$

- Al aumentar la muestra, el estimador se acerca al parámetro
- La Ley de los Grandes Números garantiza consistencia

**4. Suficiencia**
- El estimador **resume toda la información** relevante de la muestra
- No se pierde información útil

###  Estimadores comunes

| Parámetro | Estimador | Fórmula | Propiedades |
|-----------|-----------|---------|-------------|
| **Media μ** | $\bar{X}$ | $\frac{1}{n}\sum X_i$ | Insesgado, consistente |
| **Varianza σ²** | $S^2$ | $\frac{1}{n-1}\sum (X_i - \bar{X})^2$ | Insesgado |
| **Varianza σ²** | $\hat{\sigma}^2$ | $\frac{1}{n}\sum (X_i - \bar{X})^2$ | Sesgado, consistente |
| **Proporción p** | $\hat{p}$ | $\frac{\text{éxitos}}{n}$ | Insesgado, consistente |

###  Notación importante

- **Parámetro:** Valor poblacional (fijo pero desconocido)
- **Estimador:** Variable aleatoria (función de la muestra)
- **Estimación:** Valor numérico (con datos específicos)

$$\underbrace{\hat{\theta}}_{\text{estimador}} = h(X_1,\dots,X_n) \quad \xrightarrow{\text{datos}} \quad \underbrace{\hat{\theta}_{\text{obs}}}_{\text{estimación}}$$

In [2]:
# prueba de inicio para verificar el kernel y las bibliotecas necesarias

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, t
from ipywidgets import interact, widgets
import warnings
warnings.filterwarnings('ignore')



print("Kernel listo y configurado")

Kernel listo y configurado


In [ ]:
# Estimadores Puntuales - Demostración Interactiva


def demostracion_estimadores(distribucion="Normal", n_muestra=30, n_simulaciones=500, param1=0, param2=1):
    """
    Demostración de propiedades de estimadores (insesgadez, consistencia, eficiencia)
    """
    np.random.seed(42)
    
    # Configuración según la distribución
    if distribucion == "Normal":
        mu, sigma = param1, param2
        titulo = f"N({mu}, {sigma}²)"
        param_teorico = mu
        nombre_param = "Media μ"
        
        # Generar muestras
        muestras = np.random.normal(mu, sigma, size=(n_simulaciones, n_muestra))
        
    elif distribucion == "Exponencial":
        lamb = param1 if param1 > 0 else 1
        titulo = f"Exp(λ={lamb})"
        param_teorico = 1/lamb
        nombre_param = "Media μ = 1/λ"
        
        muestras = np.random.exponential(1/lamb, size=(n_simulaciones, n_muestra))
        
    else:  # Uniforme
        a, b = param1, param2 if param2 > param1 else param1 + 1
        titulo = f"U({a}, {b})"
        param_teorico = (a + b)/2
        nombre_param = "Media μ = (a+b)/2"
        
        muestras = np.random.uniform(a, b, size=(n_simulaciones, n_muestra))
    
    # Calcular diferentes estimadores
    media_muestral = np.mean(muestras, axis=1)  # Estimador 1: media
    
    if distribucion == "Normal":
        # Estimador 2: mediana (alternativa robusta)
        mediana_muestral = np.median(muestras, axis=1)
        estimadores = [media_muestral, mediana_muestral]
        nombres = ["Media", "Mediana"]
        colores = ["skyblue", "lightcoral"]
    else:
        estimadores = [media_muestral]
        nombres = ["Media"]
        colores = ["skyblue"]
    
    # Calcular sesgo y varianza para cada estimador
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Gráfico 1: Distribución de los estimadores
    for i, (est, nombre, color) in enumerate(zip(estimadores, nombres, colores)):
        axes[0].hist(est, bins=30, density=True, alpha=0.5, 
                     color=color, edgecolor='black', label=f'{nombre}')
        axes[0].axvline(param_teorico, color='red', linestyle='--', 
                       linewidth=2, label=f'Valor real = {param_teorico:.3f}')
        
        # Mostrar media del estimador
        media_est = np.mean(est)
        axes[0].axvline(media_est, color=color, linestyle=':', 
                       linewidth=2, label=f'Media({nombre}) = {media_est:.3f}')
    
    axes[0].set_title(f'Distribución de Estimadores\n{titulo}, n = {n_muestra}', fontsize=12)
    axes[0].set_xlabel('Valor del estimador')
    axes[0].set_ylabel('Densidad')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Gráfico 2: Convergencia (consistencia)
    tamaños = np.arange(5, n_muestra + 1, 5)
    sesgos = []
    varianzas = []
    
    for m in tamaños:
        if distribucion == "Normal":
            est_m = np.mean(np.random.normal(mu, sigma, size=(n_simulaciones, m)), axis=1)
        elif distribucion == "Exponencial":
            est_m = np.mean(np.random.exponential(1/lamb, size=(n_simulaciones, m)), axis=1)
        else:
            est_m = np.mean(np.random.uniform(a, b, size=(n_simulaciones, m)), axis=1)
        
        sesgos.append(np.mean(est_m) - param_teorico)
        varianzas.append(np.var(est_m))
    
    axes[1].plot(tamaños, sesgos, 'b-o', linewidth=2, markersize=4, label='Sesgo')
    axes[1].plot(tamaños, varianzas, 'r-s', linewidth=2, markersize=4, label='Varianza')
    axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
    axes[1].set_xlabel('Tamaño de muestra (n)')
    axes[1].set_ylabel('Valor')
    axes[1].set_title('Propiedades del Estimador Media\nSesgo → 0, Varianza → 0', fontsize=12)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Información
    info = f"""
    📊 PROPIEDADES DEL ESTIMADOR MEDIA
    
    • Parámetro a estimar: {nombre_param} = {param_teorico:.4f}
    • Tamaño de muestra: n = {n_muestra}
    • Número de simulaciones: {n_simulaciones}
    
    Media del estimador: {np.mean(media_muestral):.4f}
    Sesgo: {np.mean(media_muestral) - param_teorico:.6f}
    Varianza del estimador: {np.var(media_muestral):.6f}
    Error cuadrático medio (ECM): {np.mean((media_muestral - param_teorico)**2):.6f}
    """
    
    plt.suptitle('Estimadores Puntuales - Media Muestral', fontsize=14, fontweight='bold')
    fig.text(0.5, -0.02, info, ha='center', fontsize=10,
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    # Interpretación
    print("\n" + "="*60)
    print("📌 INTERPRETACIÓN DE PROPIEDADES")
    print("="*60)
    print(f"""
    ✓ Insesgadez: Sesgo = {np.mean(media_muestral) - param_teorico:.6f} ≈ 0
      → El estimador {nombre_param} acierta en promedio.
    
    ✓ Consistencia: Al aumentar n, sesgo y varianza → 0
      → Con más datos, el estimador se acerca al valor real.
    
    ✓ Eficiencia: La varianza disminuye con n
      → Mayor precisión con muestras más grandes.
    """)
    print("="*60)

# Interfaz interactiva
interact(demostracion_estimadores,
         distribucion=widgets.Dropdown(
             options=['Normal', 'Exponencial', 'Uniforme'],
             value='Normal',
             description='Distribución:',
             style={'description_width': 'initial'}
         ),
         n_muestra=widgets.IntSlider(
             min=5, max=100, step=5, value=30,
             description='Tamaño de muestra (n):',
             style={'description_width': 'initial'}
         ),
         n_simulaciones=widgets.IntSlider(
             min=100, max=1000, step=100, value=500,
             description='Simulaciones:',
             style={'description_width': 'initial'}
         ),
         param1=widgets.FloatSlider(
             min=-3, max=3, step=0.5, value=0,
             description='μ / λ / a:',
             style={'description_width': 'initial'}
         ),
         param2=widgets.FloatSlider(
             min=0.1, max=5, step=0.5, value=1,
             description='σ / b:',
             style={'description_width': 'initial'}
         ))

interactive(children=(Dropdown(description='Distribución:', options=('Normal', 'Exponencial', 'Uniforme'), sty…

<function __main__.demostracion_estimadores(distribucion='Normal', n_muestra=30, n_simulaciones=500, param1=0, param2=1)>

## Métodos de Estimación

###  Comparación de Métodos

| Característica | Máxima Verosimilitud (EMV) | Método de Momentos (MM) |
|----------------|---------------------------|------------------------|
| **Principio** | Maximizar $L(θ\|x)$ | Igualar momentos muestrales y poblacionales |
| **Fundamento** | Encontrar θ que hace los datos más probables | Aproximar momentos teóricos con empíricos |
| **Complejidad** | Puede requerir optimización numérica | Generalmente más simple |
| **Eficiencia** | Asintóticamente eficiente | Menos eficiente que EMV |
| **Sesgo** | Puede tener sesgo en muestras pequeñas | Generalmente insesgado o poco sesgado |
| **Invarianza** | Sí: $\widehat{g(θ)} = g(\hat{θ})$ | No necesariamente |

###  Función de Verosimilitud

$$L(\theta | x) = \prod_{i=1}^{n} f(x_i | \theta)$$

**Log-verosimilitud** (más fácil de maximizar):

$$\ell(\theta) = \ln L(\theta | x) = \sum_{i=1}^{n} \ln f(x_i | \theta)$$

###  EMV para distribuciones comunes

| Distribución | EMV para parámetros |
|--------------|---------------------|
| **Normal** | $\hat{\mu} = \bar{x}$, $\hat{\sigma}^2 = \frac{1}{n}\sum (x_i - \bar{x})^2$ |
| **Bernoulli** | $\hat{p} = \frac{\sum x_i}{n}$ |
| **Poisson** | $\hat{\lambda} = \bar{x}$ |
| **Exponencial** | $\hat{\lambda} = \frac{1}{\bar{x}}$ |
| **Uniforme** | $\hat{a} = \min(x_i)$, $\hat{b} = \max(x_i)$ |

###  Método de Momentos

**Principio:** Igualar momentos poblacionales y muestrales

$$ \frac{1}{n}\sum_{i=1}^{n} X_i^k = \mathbb{E}[X^k] \quad \text{para } k = 1, 2, \dots, m $$

**Ejemplo - Distribución Gamma(α, β):**

1. Primer momento: $\bar{X} = \frac{\alpha}{\beta}$
2. Segundo momento: $\frac{1}{n}\sum X_i^2 = \frac{\alpha(\alpha+1)}{\beta^2}$

Resolviendo:
$$\hat{\alpha} = \frac{\bar{X}^2}{\frac{1}{n}\sum X_i^2 - \bar{X}^2}, \quad \hat{\beta} = \frac{\bar{X}}{\frac{1}{n}\sum X_i^2 - \bar{X}^2}$$

###  Comparación práctica

```markdown
Ventajas del EMV:
✓ Asintóticamente insesgado
✓ Varianza mínima asintótica
✓ Invarianza ante transformaciones

Desventajas del EMV:
✗ Requiere resolver ecuaciones complejas
✗ Puede no tener solución cerrada
✗ Sensible a valores atípicos

Ventajas del MM:
✓ Simple de calcular
✓ Siempre tiene solución (con momentos finitos)
✓ Buen punto de partida para EMV

Desventajas del MM:
✗ Menos eficiente que EMV
✗ Puede producir estimadores fuera del espacio paramétrico

In [4]:
# prueba de inicio para verificar el kernel y las bibliotecas necesarias

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, expon, gamma
from scipy.optimize import minimize
from ipywidgets import interact, widgets
import warnings
warnings.filterwarnings('ignore')





print("Kernel listo y configurado")

Kernel listo y configurado


In [5]:

### Nivel 2: Código Interactivo
# EMV y Método de Momentos - Demostración Interactiva
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, expon, gamma
from scipy.optimize import minimize
from ipywidgets import interact, widgets
import warnings
warnings.filterwarnings('ignore')

def demostracion_estimadores_metodos(distribucion="Normal", n_muestra=50, n_simulaciones=300):
    """
    Comparación entre EMV y Método de Momentos
    """
    np.random.seed(42)
    
    if distribucion == "Normal":
        mu_real, sigma_real = 2, 1.5
        titulo = f"N(μ={mu_real}, σ={sigma_real})"
        
        # Generar muestras
        muestras = np.random.normal(mu_real, sigma_real, size=(n_simulaciones, n_muestra))
        
        # EMV
        mu_emv = np.mean(muestras, axis=1)
        sigma2_emv = np.var(muestras, axis=1, ddof=0)  # dividir por n
        sigma_emv = np.sqrt(sigma2_emv)
        
        # Método de Momentos (para Normal, igual que EMV)
        mu_mm = mu_emv
        sigma_mm = sigma_emv
        
        param_teorico_mu = mu_real
        param_teorico_sigma = sigma_real
        
        resultados = [
            ("μ (EMV/MM)", mu_emv, mu_mm, param_teorico_mu),
            ("σ (EMV/MM)", sigma_emv, sigma_mm, param_teorico_sigma)
        ]
        
    elif distribucion == "Exponencial":
        lambda_real = 0.5
        titulo = f"Exp(λ={lambda_real})"
        
        muestras = np.random.exponential(1/lambda_real, size=(n_simulaciones, n_muestra))
        
        # EMV: λ̂ = 1/̄x
        lambda_emv = 1 / np.mean(muestras, axis=1)
        
        # Método de Momentos: mismo que EMV para exponencial
        lambda_mm = lambda_emv
        
        param_teorico = lambda_real
        
        resultados = [("λ (EMV/MM)", lambda_emv, lambda_mm, param_teorico)]
        
    else:  # Gamma
        alpha_real, beta_real = 2, 0.5
        titulo = f"Gamma(α={alpha_real}, β={beta_real})"
        
        muestras = np.random.gamma(alpha_real, 1/beta_real, size=(n_simulaciones, n_muestra))
        
        # EMV (aproximación numérica)
        def log_likelihood_gamma(params, x):
            alpha, beta = params
            if alpha <= 0 or beta <= 0:
                return 1e10
            return -np.sum(gamma.logpdf(x, alpha, scale=1/beta))
        
        alpha_emv, beta_emv = [], []
        for i in range(min(n_simulaciones, 100)):  # Limitar para velocidad
            muestra = muestras[i]
            res = minimize(log_likelihood_gamma, [alpha_real, beta_real], 
                          args=(muestra,), method='L-BFGS-B',
                          bounds=[(0.01, 10), (0.01, 10)])
            if res.success:
                alpha_emv.append(res.x[0])
                beta_emv.append(res.x[1])
        
        alpha_emv = np.array(alpha_emv)
        beta_emv = np.array(beta_emv)
        
        # Método de Momentos
        mean_sample = np.mean(muestras, axis=1)
        var_sample = np.var(muestras, axis=1, ddof=0)
        
        alpha_mm = mean_sample**2 / var_sample
        beta_mm = mean_sample / var_sample
        
        resultados = [
            ("α (EMV)", alpha_emv, alpha_mm, alpha_real),
            ("β (EMV)", beta_emv, beta_mm, beta_real)
        ]
    
    # Crear gráficos
    n_resultados = len(resultados)
    fig, axes = plt.subplots(1, n_resultados, figsize=(6*n_resultados, 5))
    
    if n_resultados == 1:
        axes = [axes]
    
    for idx, (nombre, estimador_emv, estimador_mm, valor_real) in enumerate(resultados):
        # Histograma de estimadores
        axes[idx].hist(estimador_emv, bins=30, density=True, alpha=0.5,
                      color='skyblue', edgecolor='black', label='EMV')
        
        if not np.array_equal(estimador_emv, estimador_mm):
            axes[idx].hist(estimador_mm, bins=30, density=True, alpha=0.5,
                          color='lightcoral', edgecolor='black', label='Método de Momentos')
        
        axes[idx].axvline(valor_real, color='red', linestyle='--', 
                         linewidth=2, label=f'Valor real = {valor_real:.3f}')
        
        # Media de los estimadores
        media_emv = np.mean(estimador_emv)
        axes[idx].axvline(media_emv, color='blue', linestyle=':', 
                         linewidth=2, label=f'EMV = {media_emv:.3f}')
        
        if not np.array_equal(estimador_emv, estimador_mm):
            media_mm = np.mean(estimador_mm)
            axes[idx].axvline(media_mm, color='red', linestyle=':', 
                             linewidth=2, label=f'MM = {media_mm:.3f}')
        
        axes[idx].set_title(f'{nombre}\n{titulo}', fontsize=11)
        axes[idx].set_xlabel('Valor del estimador')
        axes[idx].set_ylabel('Densidad')
        axes[idx].legend(fontsize=9)
        axes[idx].grid(True, alpha=0.3)
    
    # Información
    info = f"""
    📊 COMPARACIÓN: EMV vs MÉTODO DE MOMENTOS
    
    • Distribución: {titulo}
    • Tamaño de muestra: n = {n_muestra}
    • Número de simulaciones: {n_simulaciones}
    """
    
    for nombre, estimador_emv, estimador_mm, valor_real in resultados:
        info += f"""
    {nombre}:
      Valor real: {valor_real:.4f}
      EMV: media = {np.mean(estimador_emv):.4f}, var = {np.var(estimador_emv):.6f}
      MM:  media = {np.mean(estimador_mm):.4f}, var = {np.var(estimador_mm):.6f}
    """
    
    plt.suptitle('Estimación por Máxima Verosimilitud (EMV) vs Método de Momentos (MM)', 
                fontsize=14, fontweight='bold')
    fig.text(0.5, -0.02, info, ha='center', fontsize=9,
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*60)
    print("📌 COMPARACIÓN DE MÉTODOS")
    print("="*60)
    print("""
    ✓ EMV: Mayor eficiencia asintótica (mínima varianza)
    ✓ MM:  Más simple, buen punto de partida
    ✓ Ambos son consistentes (convergen al valor real)
    
    Recomendación: Usar EMV cuando sea posible calcularlo.
    """)
    print("="*60)

# Interfaz
interact(demostracion_estimadores_metodos,
         distribucion=widgets.Dropdown(
             options=['Normal', 'Exponencial', 'Gamma'],
             value='Normal',
             description='Distribución:',
             style={'description_width': 'initial'}
         ),
         n_muestra=widgets.IntSlider(
             min=20, max=100, step=10, value=50,
             description='Tamaño de muestra (n):',
             style={'description_width': 'initial'}
         ),
         n_simulaciones=widgets.IntSlider(
             min=100, max=500, step=100, value=300,
             description='Simulaciones:',
             style={'description_width': 'initial'}
         ))

interactive(children=(Dropdown(description='Distribución:', options=('Normal', 'Exponencial', 'Gamma'), style=…

<function __main__.demostracion_estimadores_metodos(distribucion='Normal', n_muestra=50, n_simulaciones=300)>

## Intervalos de Confianza

###  Definición

Un **intervalo de confianza (IC)** es un rango de valores que contiene al parámetro poblacional con una probabilidad específica (nivel de confianza).

$$P(\hat{\theta}_L \leq \theta \leq \hat{\theta}_U) = 1 - \alpha$$

Donde:
- $1-\alpha$: **Nivel de confianza** (usualmente 0.90, 0.95, 0.99)
- $\alpha$: **Nivel de significancia**

###  Analogía

> "Es como **pescar con una red**: no sabes exactamente dónde está el pez (parámetro), pero la red (intervalo) tiene una probabilidad del 95% de atraparlo."

###  Interpretación correcta

| Interpretación INCORRECTA | Interpretación CORRECTA |
|--------------------------|------------------------|
| "Hay un 95% de probabilidad de que μ esté en [a,b]" | "El 95% de los intervalos construidos contendrán a μ" |

**Explicación:** El parámetro es fijo, el intervalo es aleatorio.

###  Intervalos comunes

| Parámetro | Intervalo de confianza (95%) | Condiciones |
|-----------|------------------------------|-------------|
| **Media μ (σ conocida)** | $\bar{x} \pm z_{\alpha/2} \frac{\sigma}{\sqrt{n}}$ | Normal o n≥30 |
| **Media μ (σ desconocida)** | $\bar{x} \pm t_{\alpha/2, n-1} \frac{s}{\sqrt{n}}$ | Normal o n≥30 |
| **Proporción p** | $\hat{p} \pm z_{\alpha/2} \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$ | n≥30, np≥5, n(1-p)≥5 |
| **Varianza σ²** | $\left[\frac{(n-1)s^2}{\chi^2_{\alpha/2, n-1}}, \frac{(n-1)s^2}{\chi^2_{1-\alpha/2, n-1}}\right]$ | Normalidad |

###  Valores críticos comunes

| Nivel confianza | $z_{\alpha/2}$ (Normal) | $t_{\alpha/2, \infty}$ |
|----------------|------------------------|----------------------|
| **90%** | 1.645 | 1.645 |
| **95%** | 1.960 | 1.960 |
| **99%** | 2.576 | 2.576 |

###  Factores que afectan la amplitud del IC

| Factor | Efecto en amplitud | Explicación |
|--------|-------------------|-------------|
| **Aumentar n** | Disminuye | Más información = más precisión |
| **Aumentar confianza** | Aumenta | Mayor seguridad = intervalo más amplio |
| **Aumentar variabilidad** | Aumenta | Datos más dispersos = menos precisión |

$$ \text{Amplitud} \propto \frac{\text{Valor crítico} \times \text{Desviación}}{\sqrt{n}} $$

In [6]:
# prueba de inicio para verificar el kernel y las bibliotecas necesarias

%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, t, chi2
from ipywidgets import interact, widgets
import warnings
warnings.filterwarnings('ignore')



print("Kernel listo y configurado")

Kernel listo y configurado


In [7]:
# Intervalos de Confianza - Demostración Interactiva

def demostracion_ic(distribucion="Normal", n_muestra=30, nivel_confianza=95, n_simulaciones=100):
    """
    Demostración de intervalos de confianza
    """
    np.random.seed(42)
    
    # Parámetros poblacionales
    mu_real = 100
    sigma_real = 15
    
    # Configuración
    alfa = 1 - nivel_confianza/100
    z_critico = norm.ppf(1 - alfa/2)
    
    # Generar muestras y calcular ICs
    limites_inf = []
    limites_sup = []
    contiene = []
    
    for i in range(n_simulaciones):
        if distribucion == "Normal":
            muestra = np.random.normal(mu_real, sigma_real, n_muestra)
        else:  # t-Student (colas más pesadas)
            muestra = np.random.standard_t(5, n_muestra) * sigma_real/2 + mu_real
        
        x_bar = np.mean(muestra)
        s = np.std(muestra, ddof=1)
        
        # IC para la media (σ desconocida)
        LI = x_bar - t.ppf(1 - alfa/2, n_muestra-1) * s / np.sqrt(n_muestra)
        LS = x_bar + t.ppf(1 - alfa/2, n_muestra-1) * s / np.sqrt(n_muestra)
        
        limites_inf.append(LI)
        limites_sup.append(LS)
        contiene.append(LI <= mu_real <= LS)
    
    limites_inf = np.array(limites_inf)
    limites_sup = np.array(limites_sup)
    contiene = np.array(contiene)
    proporcion_cobertura = np.mean(contiene) * 100
    
    # Crear gráfico
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Gráfico 1: Intervalos de confianza (primeras 50 simulaciones)
    n_mostrar = min(50, n_simulaciones)
    colores = ['green' if c else 'red' for c in contiene[:n_mostrar]]
    
    for i in range(n_mostrar):
        axes[0].plot([limites_inf[i], limites_sup[i]], [i, i], 
                    color=colores[i], linewidth=2)
    
    axes[0].axvline(x=mu_real, color='blue', linestyle='--', 
                   linewidth=2, label=f'Valor real μ = {mu_real}')
    axes[0].set_xlabel('Valor del parámetro')
    axes[0].set_ylabel('Número de simulación')
    axes[0].set_title(f'Intervalos de Confianza al {nivel_confianza}%\n(Verde = contiene, Rojo = no contiene)', fontsize=12)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Gráfico 2: Histograma de la media muestral
    medias = []
    for i in range(n_simulaciones):
        if distribucion == "Normal":
            muestra = np.random.normal(mu_real, sigma_real, n_muestra)
        else:
            muestra = np.random.standard_t(5, n_muestra) * sigma_real/2 + mu_real
        medias.append(np.mean(muestra))
    
    axes[1].hist(medias, bins=25, density=True, alpha=0.7, 
                color='skyblue', edgecolor='black')
    
    # Distribución teórica de la media
    x_vals = np.linspace(mu_real - 4*sigma_real/np.sqrt(n_muestra), 
                         mu_real + 4*sigma_real/np.sqrt(n_muestra), 200)
    axes[1].plot(x_vals, norm.pdf(x_vals, mu_real, sigma_real/np.sqrt(n_muestra)), 
                'r-', linewidth=2, label='Distribución teórica de la media')
    
    axes[1].axvline(mu_real, color='blue', linestyle='--', 
                   linewidth=2, label=f'μ real = {mu_real}')
    axes[1].set_xlabel('Valor de la media muestral')
    axes[1].set_ylabel('Densidad')
    axes[1].set_title('Distribución de la Media Muestral', fontsize=12)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Información
    info = f"""
    📊 INTERVALOS DE CONFIANZA PARA LA MEDIA (σ desconocida)
    
    • Distribución: {distribucion}
    • Tamaño de muestra: n = {n_muestra}
    • Nivel de confianza teórico: {nivel_confianza}%
    • Nivel de confianza obtenido: {proporcion_cobertura:.1f}%
    • Número de intervalos que contienen a μ: {np.sum(contiene)} / {n_simulaciones}
    """
    
    plt.suptitle('Intervalos de Confianza - Interpretación', fontsize=14, fontweight='bold')
    fig.text(0.5, -0.02, info, ha='center', fontsize=10,
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*60)
    print("📌 INTERPRETACIÓN DEL INTERVALO DE CONFIANZA")
    print("="*60)
    print(f"""
    ✓ Teóricamente, el {nivel_confianza}% de los intervalos deberían contener a μ.
    ✓ En esta simulación, {proporcion_cobertura:.1f}% de los intervalos lo contienen.
    ✓ Esto NO significa que haya una probabilidad del {nivel_confianza}% 
      de que μ esté en un intervalo específico.
    
    Recordar: μ es fijo (pero desconocido), el intervalo es aleatorio.
    """)
    print("="*60)

# Interfaz
interact(demostracion_ic,
         distribucion=widgets.Dropdown(
             options=['Normal', 't-Student (con colas pesadas)'],
             value='Normal',
             description='Distribución:',
             style={'description_width': 'initial'}
         ),
         n_muestra=widgets.IntSlider(
             min=10, max=100, step=5, value=30,
             description='Tamaño de muestra (n):',
             style={'description_width': 'initial'}
         ),
         nivel_confianza=widgets.IntSlider(
             min=80, max=99, step=1, value=95,
             description='Nivel confianza (%):',
             style={'description_width': 'initial'}
         ),
         n_simulaciones=widgets.IntSlider(
             min=50, max=200, step=10, value=100,
             description='Simulaciones:',
             style={'description_width': 'initial'}
         ))

interactive(children=(Dropdown(description='Distribución:', options=('Normal', 't-Student (con colas pesadas)'…

<function __main__.demostracion_ic(distribucion='Normal', n_muestra=30, nivel_confianza=95, n_simulaciones=100)>